In [ ]:
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification


In [ ]:
model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = TFDistilBertForSequenceClassification.from_pretrained(model_name)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_transform.bias', 'vocab_layer_norm.weight']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

In [ ]:
text = "Hello, how are you"
inputs = tokenizer(text, return_tensors="tf")


In [ ]:
outputs = model(**inputs)


In [ ]:
outputs

TFSequenceClassifierOutput(loss=None, logits=<tf.Tensor: shape=(1, 2), dtype=float32, numpy=array([[0.05252559, 0.09343087]], dtype=float32)>, hidden_states=None, attentions=None)

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
result = classifier("I love using DistilBERT in Colab!")
print(result)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9934276938438416}]


In [ ]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("cardiffnlp/tweet_sentiment_multilingual", "english")

# Determine the number of unique labels
num_labels = len(set(dataset["train"]["label"]))
print(f"Number of unique labels: {num_labels}")

# Preprocess function (simplified)
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Load tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Tokenize and preprocess dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Split dataset into train and eval subsets
train_dataset = tokenized_dataset["train"].shuffle(seed=42)
eval_dataset = tokenized_dataset["test"].shuffle(seed=42)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none"  # Disable wandb integration
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# Train model
trainer.train()


# Detect the device of the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Save model and tokenizer
model.save_pretrained("./harassment_detection_model")
tokenizer.save_pretrained("./harassment_detection_model")

# Test model on a sample text
test_text = "You're so stupid, I can't believe anyone would hire you."
inputs = tokenizer(test_text, return_tensors="pt", truncation=True, padding=True, max_length=128)

# Move input tensors to the same device as the model
inputs = {key: value.to(device) for key, value in inputs.items()}


outputs = model(**inputs)
predicted_class = torch.argmax(outputs.logits).item()

print(f"Test text: {test_text}")
print(f"Predicted class: {predicted_class}")


Number of unique labels: 3


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss


Test text: You're so stupid, I can't believe anyone would hire you.
Predicted class: 0


In [ ]:
print(set(dataset["train"]["label"]))  # Check unique labels in your dataset


{0, 1, 2}


In [ ]:
print("Logits shape:", outputs.logits.shape)  # Should be [batch_size, num_labels]
print("Labels shape:", batch["label"].shape)


NameError: name 'outputs' is not defined

In [ ]:
num_labels = len(set(dataset["train"]["label"]))
print(f"Number of unique labels: {num_labels}")



Number of unique labels: 3


In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# Load the final model
model_path = "./harassment_detection_model"
model_model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizer.from_pretrained(model_path)

# Or load a specific checkpoint
checkpoint_path = "./results/checkpoint-345"
model_checkpoint = DistilBertForSequenceClassification.from_pretrained(checkpoint_path)


In [ ]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# Load the saved model and tokenizer
model_path = "./harassment_detection_model"  # Path to your saved model folder
model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizer.from_pretrained(model_path)

# Move the model to the appropriate device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Prepare the input text
test_text = "ok i will get this done"
inputs = tokenizer(test_text, return_tensors="pt", truncation=True, padding=True, max_length=128)

# Move input tensors to the same device as the model
inputs = {key: value.to(device) for key, value in inputs.items()}

# Perform inference
with torch.no_grad():  # Disable gradient calculations for inference
    outputs = model(**inputs)
    logits = outputs.logits  # Get the raw prediction scores

# Get the predicted class
predicted_class = torch.argmax(logits).item()

print(f"Test text: {test_text}")
print(f"Predicted class: {predicted_class}")


Test text: ok i will get this done
Predicted class: 1


In [ ]:
# Prepare the input text
test_text = "saw ur presentation. epic fail. #incompetent"
inputs = tokenizer(test_text, return_tensors="pt", truncation=True, padding=True, max_length=128)

# Move input tensors to the same device as the model
inputs = {key: value.to(device) for key, value in inputs.items()}

# Perform inference
with torch.no_grad():  # Disable gradient calculations for inference
    outputs = model(**inputs)
    logits = outputs.logits  # Get the raw prediction scores

# Get the predicted class
predicted_class = torch.argmax(logits).item()

print(f"Test text: {test_text}")
print(f"Predicted class: {predicted_class}")

Test text: saw ur presentation. epic fail. #incompetent
Predicted class: 0
